# Evaluate All Trained Models on Training Data

This notebook evaluates every trained model artifact in `models/trained` against the original preprocessed training data.

It saves report-ready graphics and tables to:

`results/model_evaluation_training_data`

In [1]:
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    r"C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis"
)

TRAINED_DIR = PROJECT_ROOT / "models" / "trained"
RESULTS_DIR = PROJECT_ROOT / "results" / "model_evaluation_training_data"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"

for d in [RESULTS_DIR, FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Trained models:", TRAINED_DIR)
print("Results:", RESULTS_DIR)

Project root: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis
Trained models: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\models\trained
Results: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\model_evaluation_training_data


## Locate training data

Update `TRAINING_DATA_PATH` manually if the automatic search selects the wrong file.

In [3]:
TRAINING_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bacterial_samples.parquet"
)

if not TRAINING_DATA_PATH.exists():
    raise FileNotFoundError(TRAINING_DATA_PATH)

print(f"Using training data: {TRAINING_DATA_PATH}")

df = pd.read_parquet(TRAINING_DATA_PATH)

# Evaluate on the held-out test set
if "split" in df.columns:
    test_df = df[df["split"] == "test"].copy()

    print(f"Loaded {len(df):,} particles")
    print(f"Evaluating on {len(test_df):,} test particles")

    df = test_df
else:
    print("No split column found. Using entire dataset.")

Using training data: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\data\processed\bacterial_samples.parquet
No split column found. Using entire dataset.


## Standardize labels and feature columns

In [ ]:
def infer_label_column(df: pd.DataFrame) -> str:
    candidates = [
        "label",
        "species",
        "target",
        "class_name",
        "y",
        "y_true",
        "expected_sample",
        "sample_type",
        "bacteria_label",
        "group",
    ]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError("Could not infer label column. Set LABEL_COL manually.")

LABEL_COL = infer_label_column(df)
print("Label column:", LABEL_COL)

LABEL_NORMALIZATION = {
    "B. cereus": "B_cereus",
    "Bacillus cereus": "B_cereus",
    "bacillus_cereus": "B_cereus",
    "B_cereus": "B_cereus",

    "B. endophyticus": "B_endophyticus",
    "Bacillus endophyticus": "B_endophyticus",
    "bacillus_endophyticus": "B_endophyticus",
    "B_endophyticus": "B_endophyticus",

    "K. salsicia": "K_salsicia",
    "Kocuria salsicia": "K_salsicia",
    "kocuria_salsicia": "K_salsicia",
    "K_salsicia": "K_salsicia",

    "M. luteus": "M_luteus",
    "Micrococcus luteus": "M_luteus",
    "micrococcus": "M_luteus",
    "M_luteus": "M_luteus",

    "S. huminis": "S_huminis",
    "Staphylococcus hominis": "S_huminis",
    "staphylococcus_hominis": "S_huminis",
    "S_huminis": "S_huminis",

    "control": "control",
    "ringer": "control",
    "Ringer": "control",
}

def normalize_label(x: Any) -> str:
    if pd.isna(x):
        return "unknown"
    x = str(x)
    return LABEL_NORMALIZATION.get(x, x)

df = df.copy()
df["y_true"] = df[LABEL_COL].map(normalize_label)

df["y_true"].value_counts()

In [ ]:
def sorted_feature_cols(df: pd.DataFrame, prefix: str) -> list[str]:
    cols = [c for c in df.columns if c.startswith(prefix)]
    def key(c: str):
        tail = c.replace(prefix, "")
        try:
            return int(tail)
        except ValueError:
            return tail
    return sorted(cols, key=key)

fs_cols = sorted_feature_cols(df, "fs_")
lt_cols = sorted_feature_cols(df, "lt_")
si_cols = sorted_feature_cols(df, "si_")
scalar_cols = [c for c in ["size", "time_asymmetry"] if c in df.columns]

FEATURE_BLOCKS = {
    "fs": fs_cols,
    "lt": lt_cols,
    "si": si_cols,
    "scalar": scalar_cols,
    "fs_lt": fs_cols + lt_cols,
    "fs_scalar": fs_cols + scalar_cols,
    "lt_scalar": lt_cols + scalar_cols,
    "fs_lt_scalar": fs_cols + lt_cols + scalar_cols,
    "all": fs_cols + lt_cols + si_cols + scalar_cols,
}

{k: len(v) for k, v in FEATURE_BLOCKS.items()}

## PyTorch model definitions

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F


class DenseMultimodalSpeciesCNN(nn.Module):
    def __init__(
        self,
        n_classes: int,
        spectra_dim: int = 256,
        lifetime_dim: int = 256,
        scattering_dim: int = 1440,
        scalar_dim: int = 2,
    ):
        super().__init__()

        self.spectra_branch = nn.Sequential(
            nn.Linear(spectra_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.ReLU(),
        )

        self.lifetime_branch = nn.Sequential(
            nn.Linear(lifetime_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.ReLU(),
        )

        self.scattering_branch = nn.Sequential(
            nn.Linear(scattering_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(256, 64),
            nn.ReLU(),
        )

        self.scalar_branch = nn.Sequential(
            nn.Linear(scalar_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.Linear(208, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(64, n_classes),
        )

    def forward(self, spectra, lifetime, scattering, scalar):
        z_spec = self.spectra_branch(spectra)
        z_life = self.lifetime_branch(lifetime)
        z_scat = self.scattering_branch(scattering)
        z_scalar = self.scalar_branch(scalar)
        z = torch.cat([z_spec, z_life, z_scat, z_scalar], dim=1)
        return self.classifier(z)


class ConvBranch1D(nn.Module):
    def __init__(self, in_channels: int = 1, out_dim: int = 64, dropout: float = 0.25):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),

            nn.Dropout(dropout),
            nn.Linear(128, out_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class ScalarBranch(nn.Module):
    def __init__(self, input_dim: int = 2, out_dim: int = 16):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.BatchNorm1d(16),
            nn.Dropout(0.1),

            nn.Linear(16, out_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class RobustMultimodalClassifier(nn.Module):
    def __init__(
        self,
        n_classes: int,
        branch_dim: int = 64,
        scalar_dim: int = 16,
        modality_dropout_p: float = 0.15,
    ):
        super().__init__()

        self.modality_dropout_p = modality_dropout_p

        self.spectrometer_branch = ConvBranch1D(out_dim=branch_dim, dropout=0.25)
        self.lifetime_branch = ConvBranch1D(out_dim=branch_dim, dropout=0.25)
        self.scattering_branch = ConvBranch1D(out_dim=branch_dim, dropout=0.25)
        self.scalar_branch = ScalarBranch(input_dim=2, out_dim=scalar_dim)

        fusion_dim = branch_dim * 3 + scalar_dim

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.35),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(64, n_classes),
        )

    def _maybe_drop(self, z):
        if not self.training or self.modality_dropout_p <= 0:
            return z

        if torch.rand(1, device=z.device).item() < self.modality_dropout_p:
            return torch.zeros_like(z)

        return z

    def forward(self, batch, enabled_modalities=None):
        enabled = set(enabled_modalities or ["spectrometer", "lifetime", "scattering", "scalar"])

        z_spec = self.spectrometer_branch(batch["spectrometer"])
        z_life = self.lifetime_branch(batch["lifetime"])
        z_scat = self.scattering_branch(batch["scattering"])
        z_scalar = self.scalar_branch(batch["scalar"])

        if "spectrometer" not in enabled:
            z_spec = torch.zeros_like(z_spec)
        if "lifetime" not in enabled:
            z_life = torch.zeros_like(z_life)
        if "scattering" not in enabled:
            z_scat = torch.zeros_like(z_scat)
        if "scalar" not in enabled:
            z_scalar = torch.zeros_like(z_scalar)

        z_spec = self._maybe_drop(z_spec)
        z_life = self._maybe_drop(z_life)
        z_scat = self._maybe_drop(z_scat)
        z_scalar = self._maybe_drop(z_scalar)

        z = torch.cat([z_spec, z_life, z_scat, z_scalar], dim=1)
        return self.classifier(z)

## Evaluation helpers

In [ ]:
def get_state_dict(ckpt: Any) -> dict:
    if isinstance(ckpt, dict):
        for key in ["model_state_dict", "state_dict"]:
            if key in ckpt:
                return ckpt[key]
    return ckpt


def class_names_from_checkpoint(ckpt: Any) -> list[str]:
    if isinstance(ckpt, dict):
        if "class_names" in ckpt:
            return [normalize_label(x) for x in ckpt["class_names"]]
        if "classes" in ckpt:
            return [normalize_label(x) for x in ckpt["classes"]]
        if "label_encoder" in ckpt:
            return [normalize_label(x) for x in ckpt["label_encoder"].classes_]
        if "n_classes" in ckpt:
            return [f"class_{i}" for i in range(int(ckpt["n_classes"]))]
    return ["B_cereus", "B_endophyticus", "K_salsicia", "M_luteus", "S_huminis"]


def find_scaler(model_dir: Path, names: list[str]):
    for name in names:
        p = model_dir / name
        if p.exists():
            return joblib.load(p)
    return None


def apply_scaler_if_present(X: np.ndarray, scaler):
    if scaler is None:
        return X.astype(np.float32)
    return scaler.transform(X).astype(np.float32)


def model_name_from_path(path: Path) -> str:
    if path.name in {"model.pt", "model.joblib"}:
        return path.parent.name
    return path.stem


def discover_model_files(trained_dir: Path) -> list[Path]:
    return sorted([p for p in trained_dir.rglob("*") if p.name in {"model.pt", "model.joblib"}])


def select_feature_block_for_sklearn(model, feature_blocks: dict[str, list[str]]):
    expected = getattr(model, "n_features_in_", None)
    if expected is None:
        raise ValueError("Sklearn model has no n_features_in_.")
    matches = [(name, cols) for name, cols in feature_blocks.items() if len(cols) == expected]
    if not matches:
        raise ValueError(
            f"No feature block matches expected feature count {expected}. "
            f"Available: { {k: len(v) for k, v in feature_blocks.items()} }"
        )
    return matches[0]


def infer_torch_architecture(state_dict: dict) -> str:
    keys = list(state_dict.keys())
    if any(k.startswith("spectrometer_branch.") for k in keys):
        return "robust_cnn"
    if any(k.startswith("spectra_branch.") for k in keys):
        return "dense_multimodal"
    raise ValueError("Could not infer PyTorch architecture from checkpoint keys.")


def build_torch_model_from_checkpoint(ckpt: Any):
    state_dict = get_state_dict(ckpt)
    class_names = class_names_from_checkpoint(ckpt)
    n_classes = len(class_names)

    arch = infer_torch_architecture(state_dict)

    if arch == "robust_cnn":
        model = RobustMultimodalClassifier(n_classes=n_classes)
    elif arch == "dense_multimodal":
        model = DenseMultimodalSpeciesCNN(n_classes=n_classes)
    else:
        raise ValueError(f"Unsupported architecture: {arch}")

    model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model, arch, class_names


def build_tabular_inputs(df: pd.DataFrame, model_dir: Path):
    X_spec = df[fs_cols].to_numpy(dtype=np.float32)
    X_life = df[lt_cols].to_numpy(dtype=np.float32)
    X_scat = df[si_cols].to_numpy(dtype=np.float32)
    X_scalar = df[scalar_cols].to_numpy(dtype=np.float32)

    spec_scaler = find_scaler(model_dir, ["spectrometer_scaler.joblib", "spectra_scaler.joblib"])
    life_scaler = find_scaler(model_dir, ["lifetime_scaler.joblib"])
    scat_scaler = find_scaler(model_dir, ["scattering_scaler.joblib"])
    scalar_scaler = find_scaler(model_dir, ["scalar_scaler.joblib"])

    X_spec = apply_scaler_if_present(X_spec, spec_scaler)
    X_life = apply_scaler_if_present(X_life, life_scaler)
    X_scat = apply_scaler_if_present(X_scat, scat_scaler)
    X_scalar = apply_scaler_if_present(X_scalar, scalar_scaler)

    return X_spec, X_life, X_scat, X_scalar


def predict_torch_model(model, arch: str, model_dir: Path, df: pd.DataFrame, class_names: list[str], batch_size: int = 1024):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    X_spec, X_life, X_scat, X_scalar = build_tabular_inputs(df, model_dir)

    preds = []
    probas = []

    with torch.no_grad():
        for start in range(0, len(df), batch_size):
            end = min(start + batch_size, len(df))

            spec = torch.tensor(X_spec[start:end], dtype=torch.float32, device=device)
            life = torch.tensor(X_life[start:end], dtype=torch.float32, device=device)
            scat = torch.tensor(X_scat[start:end], dtype=torch.float32, device=device)
            scalar = torch.tensor(X_scalar[start:end], dtype=torch.float32, device=device)

            if arch == "robust_cnn":
                batch = {
                    "spectrometer": spec.unsqueeze(1),
                    "lifetime": life.unsqueeze(1),
                    "scattering": scat.unsqueeze(1),
                    "scalar": scalar,
                }
                logits = model(batch)
            else:
                logits = model(spec, life, scat, scalar)

            proba = F.softmax(logits, dim=1).detach().cpu().numpy()
            idx = proba.argmax(axis=1)

            probas.append(proba)
            preds.extend([class_names[i] for i in idx])

    return np.array(preds), np.vstack(probas)


def find_label_encoder(model_dir: Path):
    candidates = [
        model_dir / "label_encoder.joblib",
        model_dir.parent / "label_encoder.joblib",
    ]
    for p in candidates:
        if p.exists():
            return joblib.load(p)
    return None


def predict_sklearn_model(model_path: Path, df: pd.DataFrame):
    model = joblib.load(model_path)
    if not hasattr(model, "predict"):
        raise TypeError(f"{model_path} is not a predictive model.")

    block_name, cols = select_feature_block_for_sklearn(model, FEATURE_BLOCKS)
    X = df[cols].to_numpy(dtype=np.float32)

    pred = model.predict(X)
    le = find_label_encoder(model_path.parent)
    if le is not None:
        try:
            pred = le.inverse_transform(pred)
        except Exception:
            pass

    pred = np.array([normalize_label(x) for x in pred])

    proba = model.predict_proba(X) if hasattr(model, "predict_proba") else None
    return pred, proba, block_name


def compute_metrics(model_id: str, y_true, y_pred, proba=None):
    y_true = np.array([normalize_label(x) for x in y_true])
    y_pred = np.array([normalize_label(x) for x in y_pred])

    row = {
        "model_id": model_id,
        "n_samples": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "unknown_fraction": float(np.mean(y_pred == "unknown")),
    }

    if proba is not None:
        row["mean_confidence"] = float(np.max(proba, axis=1).mean())
        row["median_confidence"] = float(np.median(np.max(proba, axis=1)))
    else:
        row["mean_confidence"] = np.nan
        row["median_confidence"] = np.nan

    return row

## Evaluate all trained models

In [ ]:
model_files = discover_model_files(TRAINED_DIR)

print(f"Found {len(model_files)} candidate model files:")
for p in model_files:
    print(" ", p)

In [ ]:
all_metrics = []
all_predictions = []
all_class_reports = []
failures = []

y_true = df["y_true"].to_numpy()

for model_path in model_files:
    model_id = model_name_from_path(model_path)
    print("\n" + "=" * 100)
    print("Evaluating:", model_id)
    print("Path:", model_path)

    try:
        if model_path.suffix == ".joblib":
            pred, proba, feature_block = predict_sklearn_model(model_path, df)
            model_type = "sklearn"
            architecture = feature_block

        elif model_path.suffix == ".pt":
            ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
            model, architecture, class_names = build_torch_model_from_checkpoint(ckpt)
            pred, proba = predict_torch_model(model, architecture, model_path.parent, df, class_names)
            model_type = "torch"

        else:
            continue

        metrics = compute_metrics(model_id, y_true, pred, proba)
        metrics["model_type"] = model_type
        metrics["architecture"] = architecture
        metrics["model_path"] = str(model_path)
        all_metrics.append(metrics)

        pred_frame = pd.DataFrame({
            "model_id": model_id,
            "model_type": model_type,
            "architecture": architecture,
            "y_true": y_true,
            "y_pred": pred,
        })

        if proba is not None:
            pred_frame["confidence"] = np.max(proba, axis=1)

        for col in ["raw_file", "source_file", "particle_index", "experiment_name"]:
            if col in df.columns:
                pred_frame[col] = df[col].values

        all_predictions.append(pred_frame)

        labels_for_report = sorted(pd.unique(np.concatenate([y_true, pred])))
        report = classification_report(
            y_true,
            pred,
            labels=labels_for_report,
            output_dict=True,
            zero_division=0,
        )

        report_df = (
            pd.DataFrame(report)
            .T
            .reset_index()
            .rename(columns={"index": "class_label"})
        )
        report_df["model_id"] = model_id
        all_class_reports.append(report_df)

        print(
            f"accuracy={metrics['accuracy']:.4f} | "
            f"balanced_accuracy={metrics['balanced_accuracy']:.4f} | "
            f"macro_f1={metrics['macro_f1']:.4f}"
        )

    except Exception as exc:
        print("FAILED:", repr(exc))
        failures.append({
            "model_id": model_id,
            "model_path": str(model_path),
            "error": repr(exc),
        })

metrics_df = pd.DataFrame(all_metrics).sort_values("balanced_accuracy", ascending=False) if all_metrics else pd.DataFrame()
predictions_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
class_report_df = pd.concat(all_class_reports, ignore_index=True) if all_class_reports else pd.DataFrame()
failures_df = pd.DataFrame(failures)

metrics_df

## Save report-ready tables

In [ ]:
metrics_path = TABLES_DIR / "training_model_comparison_metrics.csv"
predictions_path = TABLES_DIR / "training_model_predictions.parquet"
class_report_path = TABLES_DIR / "training_per_class_metrics.csv"
failures_path = TABLES_DIR / "training_model_failures.csv"

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_parquet(predictions_path, index=False)
class_report_df.to_csv(class_report_path, index=False)
failures_df.to_csv(failures_path, index=False)

print("Saved:")
print(metrics_path)
print(predictions_path)
print(class_report_path)
print(failures_path)

## Model comparison table

In [ ]:
display_cols = [
    "model_id",
    "model_type",
    "architecture",
    "n_samples",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "mean_confidence",
    "unknown_fraction",
]

metrics_df[display_cols].style.format({
    "accuracy": "{:.3f}",
    "balanced_accuracy": "{:.3f}",
    "macro_f1": "{:.3f}",
    "weighted_f1": "{:.3f}",
    "mean_confidence": "{:.3f}",
    "unknown_fraction": "{:.3f}",
})

In [ ]:
prediction_distribution = (
    predictions_df
    .groupby(["model_id", "y_pred"])
    .size()
    .reset_index(name="n")
)

prediction_distribution["fraction"] = (
    prediction_distribution["n"]
    / prediction_distribution.groupby("model_id")["n"].transform("sum")
)

prediction_distribution_path = TABLES_DIR / "training_prediction_distribution.csv"
prediction_distribution.to_csv(prediction_distribution_path, index=False)

prediction_distribution.head(20)

## Report graphics: model metric comparison

In [ ]:
def save_bar_chart(metric: str, title: str):
    plot_df = metrics_df.sort_values(metric, ascending=True)

    fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(plot_df))))
    ax.barh(plot_df["model_id"], plot_df[metric])
    ax.set_xlabel(metric.replace("_", " ").title())
    ax.set_ylabel("Model")
    ax.set_title(title)
    ax.set_xlim(0, 1.0)
    fig.tight_layout()

    out = FIGURES_DIR / f"training_{metric}_bar_chart.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

for metric in ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]:
    save_bar_chart(metric, f"Training Data Model Comparison: {metric.replace('_', ' ').title()}")

## Report graphics: normalized confusion matrices

In [ ]:
def plot_confusion_matrix_for_model(model_id: str):
    sub = predictions_df[predictions_df["model_id"] == model_id]
    labels = sorted(pd.unique(pd.concat([sub["y_true"], sub["y_pred"]], ignore_index=True)))

    cm = confusion_matrix(sub["y_true"], sub["y_pred"], labels=labels)
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm_norm, aspect="auto")
    ax.set_title(f"Normalized Confusion Matrix: {model_id}")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)

    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(
                j,
                i,
                f"{cm_norm[i, j]:.2f}\n({cm[i, j]})",
                ha="center",
                va="center",
                fontsize=8,
            )

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()

    safe_model_id = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in model_id)
    out = FIGURES_DIR / f"training_confusion_matrix_{safe_model_id}.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

top_models = metrics_df.sort_values("balanced_accuracy", ascending=False)["model_id"].head(8).tolist()

for model_id in top_models:
    plot_confusion_matrix_for_model(model_id)

## Report graphics: predicted class distributions

In [ ]:
for model_id in metrics_df["model_id"].tolist():
    sub = prediction_distribution[prediction_distribution["model_id"] == model_id].sort_values("fraction")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(sub["y_pred"], sub["fraction"])
    ax.set_xlim(0, 1)
    ax.set_xlabel("Fraction of predictions")
    ax.set_ylabel("Predicted label")
    ax.set_title(f"Training Prediction Distribution: {model_id}")
    fig.tight_layout()

    safe_model_id = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in model_id)
    out = FIGURES_DIR / f"training_prediction_distribution_{safe_model_id}.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

## Report graphics: confidence distributions

In [ ]:
if "confidence" in predictions_df.columns:
    for model_id in metrics_df["model_id"].tolist():
        sub = predictions_df[
            (predictions_df["model_id"] == model_id)
            & predictions_df["confidence"].notna()
        ]

        if sub.empty:
            continue

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(sub["confidence"], bins=30)
        ax.set_xlabel("Prediction confidence")
        ax.set_ylabel("Particle count")
        ax.set_title(f"Training Confidence Distribution: {model_id}")
        fig.tight_layout()

        safe_model_id = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in model_id)
        out = FIGURES_DIR / f"training_confidence_distribution_{safe_model_id}.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print("Saved:", out)

## Short report summary

In [ ]:
if not metrics_df.empty:
    best = metrics_df.iloc[0]

    summary_text = (
        "Training-data model evaluation summary\n\n"
        f"The best-performing model on the training data was {best['model_id']}, "
        f"with an accuracy of {best['accuracy']:.3f}, balanced accuracy of "
        f"{best['balanced_accuracy']:.3f}, and macro F1 score of {best['macro_f1']:.3f}. "
        f"Across all evaluated models, balanced accuracy ranged from "
        f"{metrics_df['balanced_accuracy'].min():.3f} to {metrics_df['balanced_accuracy'].max():.3f}. "
        "These values should be interpreted as training-set performance rather than "
        "generalization performance; high training metrics may indicate that the model has "
        "learned the training distribution well, but final model claims should rely on "
        "held-out validation, test, and live-experiment evaluation."
    )

    summary_path = RESULTS_DIR / "training_model_evaluation_summary.txt"
    summary_path.write_text(summary_text, encoding="utf-8")

    print(summary_text)
    print("\nSaved:", summary_path)
else:
    print("No models were successfully evaluated.")

## Failures

In [ ]:
failures_df